In [16]:
from ccka.models.kernel import KernelModel
from ccka.circuits.angleEmbeddingKernel import quackEmbeddingCircuit
from ccka.aligner.kta import centroidBasedKTA
import pennylane as qml
import jax
import jax.numpy as jnp

In [17]:
data = jnp.load('../data/checkerboard_dataset.npy', allow_pickle=True).item()
X = jnp.asarray(data['x_train'])
y = jnp.asarray(data['y_train'])
x_test = jnp.asarray(data['x_test'])
y_test = jnp.asarray(data['y_test'])

X_combined = jnp.concatenate([X, x_test], axis=0)
y_combined = jnp.concatenate([y, y_test], axis=0)

N, D = X_combined.shape
x1 = jnp.repeat(X_combined, N, axis=0)      # (N*N, D)
x2 = jnp.tile(X_combined, (N, 1))

In [18]:
kernel = quackEmbeddingCircuit(
                                num_qubits = 5,
                                reps = 6,
                                reupload = True
                        )
init_weights = kernel.init_weights()
model = KernelModel(circuit = kernel)

In [19]:
aligner = centroidBasedKTA(
                    kernel_model= model,
                    data = X_combined,
                    labels = y_combined,
                    matrix_type='regular', #
                    split_size=0.5,
                    centroids= 4,
                    landmark_points=2,
                    learning_rate=0.1,
                    centroid_learning_rate=0.01,
                    subcentroid_learning_rate=0.01,
                    lambda_co=0.0,
                    lambda_kao=0.0,
                    optimizer= 'adam',
                    epochs=20,
                    eps=0.01,
                    alpha=0.1
)

In [23]:
acc = []
alpha = [0.001, 0.01, 0.1, 0.2, 0.5]
for i in range(1):

    kernel = quackEmbeddingCircuit(
                                num_qubits = 5,
                                reps = 6,
                                reupload = True
                        )
    init_weights = kernel.init_weights()
    model = KernelModel(circuit = kernel)

    aligner = centroidBasedKTA(
                    kernel_model= model,
                    data = X_combined,
                    labels = y_combined,
                    matrix_type='nystrom', #
                    split_size=0.5,
                    centroids= 4,
                    landmark_points=10,
                    lambda_co=0.0,
                    lambda_kao=0.0,
                    epochs=20,
                    eps=0.01,
                    alpha=0.01
    )

    history = aligner.align()
    acc.append(history['test_accuracy_history'][-1])

[CentroidBasedKTA] KTA alignment: 100%|██████████| 20/20 [00:09<00:00,  2.02it/s]


In [24]:
acc

[0.9]

In [25]:
history['circuit_executions']

151700